In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import boto3
import sys
from sliderule import icesat2

### Configuration

### Utility Functions

In [ ]:
s3 = boto3.client("s3")

# Display Raw
def display(s):
    sys.stdout.write(s)
    sys.stdout.flush()

# List S3 Bucket
def list_bucket(bucket, subfolder):
    resources = []
    is_truncated = True
    continuation_token = None
    while is_truncated:
        if continuation_token: response = s3.list_objects_v2(Bucket=bucket, Prefix=subfolder, ContinuationToken=continuation_token)
        else: response = s3.list_objects_v2(Bucket=bucket, Prefix=subfolder)
        display("#")
        # parse contents
        if 'Contents' in response:
            for obj in response['Contents']:
                resources.append(obj['Key'].split("/")[-1])
        # check if more data is available
        is_truncated = response['IsTruncated']
        continuation_token = response.get('NextContinuationToken')
    display("\n")
    return resources

### Read Input

In [ ]:
bucket = "sliderule-public"
subfolder = "atl24r3/parquet"
granules = list_bucket(bucket, subfolder)
print(f"Read {len(granules)} granules")

NameError: name 'gpd' is not defined

In [ ]:
# list granules
granules


In [ ]:
# configure which granule to read
selected_granule = 1

# read granule into dataframe
granule = f"s3://{bucket}/{subfolder}/{granules[selected_granule]}"
gdf = gpd.read_parquet(granule)


In [ ]:
# configure which track of granule to plot
selected_ground_track = icesat2.GT1L

# select the track
gt = gdf[gdf["gt"] == selected_ground_track]

### Display Contents

In [ ]:
gt

### Plot Photon Classifications

In [ ]:
# class_ph label + color mapping
class_labels = {
    0:  ("unclassified", "#999999"),
    1:  ("other",        "#9467bd"),
    2:  ("ground",       "#2ca02c"),
    40: ("bathymetry",   "#d62728"),
    41: ("sea surface",  "#1f77b4"),
}

fig, ax = plt.subplots(figsize=(14, 6))

for cval, (label, color) in class_labels.items():
    sub = gt[gt["class_ph"] == cval]
    if len(sub) == 0:
        continue
    ax.scatter(sub["x_atc"], sub["geoid_corr_h"],
               s=2, c=color, label=f"{cval}: {label}")

ax.set_xlabel("x_atc (m)")
ax.set_ylabel("ortho_h (m)")
ax.set_title("GT1L — ortho_h vs x_atc colored by class_ph")
ax.legend(markerscale=4, loc="best")
plt.tight_layout()
plt.show()

### Plot Uncertainties

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# sigma_tvu is a positive uncertainty magnitude; scale the colormap to a
# robust range of the actual data so the variation is visible
vmin = np.nanpercentile(gt["sigma_tvu"], 1)
vmax = np.nanpercentile(gt["sigma_tvu"], 99)

sc = ax.scatter(gt["x_atc"], gt["geoid_corr_h"],
                s=2, c=gt["sigma_tvu"],
                cmap="viridis", vmin=vmin, vmax=vmax)

cbar = fig.colorbar(sc, ax=ax, extend="both")
cbar.set_label("sigma_tvu (m)")

ax.set_xlabel("x_atc (m)")
ax.set_ylabel("ortho_h (m)")
ax.set_title("GT1L — ortho_h vs x_atc colored by sigma_tvu")
plt.tight_layout()
plt.show()